# Risk Score EDA: direction check and experimental cutoff

This notebook tests whether the source score is associated with observed loan debt groups. It is exploratory analysis for a personal project, not a validated credit-risk model.

## Workflow
1. Load customer scores, loan accounts, and debt-history records.
2. Inspect the score distribution and missingness.
3. Join loans to customers. Build two observed outcomes: worst debt group ever and debt group on the latest date. Customers without loan history remain missing; they are not assumed safe.
4. Check direction and strength using group summaries, Spearman correlation, AUC, and bad-debt rates by score bands.
5. Select an experimental cutoff using Youden's J against the independent historical debt outcome; compare proposed cutoff 446 with old cutoff 491.

## Youden's J, step by step
The hypothesis tested is **lower score means higher risk**. At each candidate cutoff, predict high risk when `risk_score <= cutoff`; the observed positive outcome is worst debt group >= 3.
- **TP**: group-3+ customer correctly flagged.
- **FN**: group-3+ customer not flagged.
- **FP**: group-1/2 customer incorrectly flagged.
- **TN**: group-1/2 customer not flagged.
- **TPR / sensitivity** = TP / (TP + FN): share of bad-debt cases caught.
- **FPR** = FP / (FP + TN): share of non-bad-debt cases incorrectly flagged.
- **Youden's J** = TPR - FPR = sensitivity + specificity - 1. The cutoff maximizing J balances catching positives against false alarms. J near 0 means weak separation; having a mathematical maximum does not make a useful model.

## Limitations
Debt group is a historical proxy, not a confirmed future-default label. Only customers joined to loan-balance records can be evaluated. The same sample selects and scores the cutoff, so results are optimistic; validate on a later time period or independent sample before relying on it.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import mannwhitneyu, spearmanr
from sklearn.metrics import roc_auc_score, roc_curve

PATH = Path(r"C:\Project\BANKING-CUSTOMER-SEGMENTATION-PROJECT\dataset\KPIM Bank Data.xlsx")

customers = pd.read_excel(
    PATH,
    sheet_name="master_khach_hang",
    usecols=["CIF", "diem_rui_ro"],
).rename(columns={"CIF": "customer_id", "diem_rui_ro": "risk_score"})
loan_accounts = pd.read_excel(
    PATH,
    sheet_name="tk_no",
    usecols=["so_tai_khoan", "CIF"],
).rename(columns={"so_tai_khoan": "account_number", "CIF": "customer_id"})
loan_balances = pd.read_excel(
    PATH,
    sheet_name="fact_du_no",
    usecols=["ngay", "so_tai_khoan", "nhom_no"],
).rename(
    columns={
        "ngay": "balance_date",
        "so_tai_khoan": "account_number",
        "nhom_no": "debt_group",
    }
)

# Excel may expose integer identifiers as floats; normalize both sides before joining.
def normalize_id(values: pd.Series) -> pd.Series:
    return values.astype("string").str.strip().str.replace(r"\.0$", "", regex=True).replace("", pd.NA)

customers["customer_id"] = normalize_id(customers["customer_id"])
loan_accounts["customer_id"] = normalize_id(loan_accounts["customer_id"])
loan_accounts["account_number"] = normalize_id(loan_accounts["account_number"])
loan_balances["account_number"] = normalize_id(loan_balances["account_number"])
loan_balances["balance_date"] = pd.to_datetime(loan_balances["balance_date"], errors="coerce")
loan_balances["debt_group"] = pd.to_numeric(loan_balances["debt_group"], errors="coerce")
customers["risk_score"] = pd.to_numeric(customers["risk_score"], errors="coerce")

print(f"Customers: {len(customers):,}; unique IDs: {customers['customer_id'].nunique():,}")
print(f"Loan accounts: {loan_accounts['account_number'].nunique():,} unique accounts")
print(f"Loan balance rows: {len(loan_balances):,}")
print(f"Balance date range: {loan_balances['balance_date'].min()} to {loan_balances['balance_date'].max()}")
print(f"Risk score coverage: {customers['risk_score'].notna().sum():,}/{len(customers):,}")

## Step 1 - Load and normalize join keys
Rename source fields to English, trim IDs while preserving alphanumeric account numbers, then report row counts, date range, and score coverage.

In [ ]:
print("Risk score distribution")
print(customers["risk_score"].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(customers["risk_score"].dropna(), bins=30, ax=axes[0])
axes[0].set(title="Risk score distribution", xlabel="Risk score", ylabel="Customer count")
sns.boxplot(x=customers["risk_score"], ax=axes[1])
axes[1].set(title="Risk score spread", xlabel="Risk score")
plt.tight_layout()
plt.show()

## Step 2 - Inspect the score distribution
Summary statistics and plots show the score range, median, and spread. A plausible-looking distribution does not prove that the score measures risk.

In [ ]:
# Debt group is a proxy outcome, not a confirmed default label. Group 3+ is treated
# as higher-risk only for the purpose of checking score direction.
valid_balances = loan_balances.dropna(subset=["account_number", "balance_date", "debt_group"]).copy()
if valid_balances.empty:
    raise ValueError("No valid loan-balance rows are available for risk-score analysis.")

as_of_date = valid_balances["balance_date"].max()
worst_debt_by_account = valid_balances.groupby("account_number")["debt_group"].max().rename("worst_debt_group")
snapshot_debt_by_account = (
    valid_balances.loc[valid_balances["balance_date"].eq(as_of_date)]
    .groupby("account_number")["debt_group"]
    .max()
    .rename("snapshot_debt_group")
)
account_outcomes = pd.concat([worst_debt_by_account, snapshot_debt_by_account], axis=1)
account_map = loan_accounts.dropna(subset=["customer_id", "account_number"]).drop_duplicates(
    subset=["customer_id", "account_number"]
)
customer_outcomes = (
    account_map.merge(account_outcomes, on="account_number", how="inner")
    .groupby("customer_id", as_index=False)
    .agg(
        worst_debt_group=("worst_debt_group", "max"),
        snapshot_debt_group=("snapshot_debt_group", "max"),
        loan_account_count=("account_number", "nunique"),
        snapshot_account_count=("snapshot_debt_group", "count"),
    )
)
analysis = customers.merge(customer_outcomes, on="customer_id", how="left", validate="one_to_one")
analysis["bad_debt_ever"] = analysis["worst_debt_group"].ge(3).where(analysis["worst_debt_group"].notna())
analysis["bad_debt_snapshot"] = analysis["snapshot_debt_group"].ge(3).where(
    analysis["snapshot_debt_group"].notna()
)

print(f"Latest loan-balance snapshot: {as_of_date:%Y-%m-%d}")
print(f"Customers with a historical loan-balance outcome: {analysis['worst_debt_group'].notna().sum():,}")
print(f"Customers represented in that exact snapshot: {analysis['snapshot_debt_group'].notna().sum():,}")
print("\nHistorical worst debt group: score by observed group")
worst_group_summary = (
    analysis.dropna(subset=["risk_score", "worst_debt_group"])
    .groupby("worst_debt_group", observed=True)
    .agg(
        customers=("customer_id", "size"),
        median_score=("risk_score", "median"),
        mean_score=("risk_score", "mean"),
        score_q1=("risk_score", lambda values: values.quantile(0.25)),
        score_q3=("risk_score", lambda values: values.quantile(0.75)),
        share_score_at_or_below_446=("risk_score", lambda values: values.le(446).mean()),
    )
)
print(worst_group_summary.round(3).to_string())


def assess_score_direction(outcome_column: str, label: str) -> dict[str, float | int]:
    sample = analysis.dropna(subset=["risk_score", outcome_column]).copy()
    bad = sample[outcome_column].ge(3)
    rho, rho_p = spearmanr(sample["risk_score"], sample[outcome_column])
    if bad.nunique() < 2:
        auc_high = np.nan
        auc_low = np.nan
        mw_p = np.nan
    else:
        auc_high = roc_auc_score(bad, sample["risk_score"])
        auc_low = roc_auc_score(bad, -sample["risk_score"])
        mw_p = mannwhitneyu(
            sample.loc[bad, "risk_score"],
            sample.loc[~bad, "risk_score"],
            alternative="two-sided",
        ).pvalue
    threshold_rates = (
        sample.assign(score_band=np.where(sample["risk_score"].le(446), "<= 446", "> 446"))
        .groupby("score_band", observed=True)[outcome_column]
        .agg(customers="size", share_group_3_plus=lambda values: values.ge(3).mean())
    )
    print(f"\n{label}: n={len(sample):,}; group 3+={int(bad.sum()):,} ({bad.mean():.1%})")
    print(f"Spearman(score, debt group)={rho:.3f}, p={rho_p:.4g}")
    print(f"AUC using higher score as riskier={auc_high:.3f}; using lower score as riskier={auc_low:.3f}")
    print(f"Mann-Whitney p-value (group 3+ vs groups 1-2)={mw_p:.4g}")
    print("Historical group-3+ rates by proposed 446 score cutoff:")
    print(threshold_rates.round(3).to_string())
    return {
        "sample_size": len(sample),
        "bad_count": int(bad.sum()),
        "spearman_rho": float(rho),
        "spearman_p": float(rho_p),
        "auc_higher_score": float(auc_high),
        "auc_lower_score": float(auc_low),
    }


historical_evidence = assess_score_direction("worst_debt_group", "Historical worst debt group")
snapshot_evidence = assess_score_direction("snapshot_debt_group", "Latest exact-date snapshot")

history = analysis.dropna(subset=["risk_score", "worst_debt_group"]).copy()
history["score_decile"] = pd.qcut(history["risk_score"], q=10, duplicates="drop")
history["bad_debt_ever"] = history["worst_debt_group"].ge(3)
decile_summary = (
    history.groupby("score_decile", observed=True)
    .agg(
        customers=("customer_id", "size"),
        median_score=("risk_score", "median"),
        bad_debt_rate=("bad_debt_ever", "mean"),
        median_worst_group=("worst_debt_group", "median"),
    )
)
print("\nHistorical group 3+ rate by ascending risk-score decile:")
print(decile_summary.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.boxplot(data=history, x="worst_debt_group", y="risk_score", showfliers=False, ax=axes[0])
axes[0].set(title="Risk score by historical worst debt group", xlabel="Worst debt group", ylabel="Risk score")
axes[1].plot(decile_summary["median_score"], decile_summary["bad_debt_rate"], marker="o")
axes[1].set(
    title="Group 3+ rate by risk-score decile",
    xlabel="Median risk score in decile",
    ylabel="Share with historical group 3+",
)
axes[1].set_ylim(bottom=0)
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

print("\nInterpretation guide: AUC > 0.5 means higher scores rank group-3+ customers higher; AUC < 0.5 means lower scores do.")
print("Treat this as directional evidence only: debt group is a proxy, not a verified default outcome, and group counts are uneven.")

## Steps 3-4 - Build observed outcomes and test direction
Analyze customers linkable to loan history. Group 3+ is used as a risk proxy. Spearman checks monotonic association; AUC compares whether high or low scores rank group-3+ customers better. AUC near 0.5 or Spearman rho near 0 indicates weak separation.

## Step 5 - Find a cutoff with Youden's J
Negate the score before constructing the ROC curve because the tested hypothesis is lower score means higher risk. Keep the positive outcome fixed as historical worst debt group >= 3. The next cell prints TP/FP/FN/TN, TPR, FPR, precision, and J for the selected cutoff and the 446/491 benchmarks.

If multiple cutoffs tie, treat the selected value as a sample-based choice, not a natural boundary between customers.

### Result on this workbook run
The historical sample contains 574 customers with debt outcomes, including 59 with worst debt group >= 3. The maximum in-sample Youden J selects cutoff 446: TPR 37.3%, FPR 27.4%, J 0.099, precision 13.5%. Cutoff 491 gives TPR 44.1%, FPR 35.3%, J 0.087, precision 12.5%. Thus 446 is the mathematical best of the tested score cutoffs for this proxy outcome, but both cutoffs have weak separation. Treat 446 as an experimental project rule, not a validated risk boundary.

In [ ]:
# Select a cutoff with Youden's J against an independent outcome. Reusing score-based
# labels would make the cutoff circular.
threshold_data = analysis.dropna(subset=["risk_score", "worst_debt_group"]).copy()
y_true = threshold_data["worst_debt_group"].ge(3).astype(int)
if y_true.nunique() != 2:
    raise ValueError("Youden cutoff requires both positive and negative debt-outcome cases.")

# The current rule treats lower scores as riskier, so negate score for ROC analysis.
fpr_values, tpr_values, roc_thresholds = roc_curve(
    y_true,
    -threshold_data["risk_score"],
    drop_intermediate=False,
)
youden_values = tpr_values - fpr_values
finite_thresholds = np.isfinite(roc_thresholds)
optimal_index = np.flatnonzero(finite_thresholds)[
    np.argmax(youden_values[finite_thresholds])
]
optimal_cutoff = -roc_thresholds[optimal_index]


def cutoff_metrics(cutoff: float) -> dict[str, float | int]:
    predicted_positive = threshold_data["risk_score"].le(cutoff)
    true_positive = int((predicted_positive & y_true.eq(1)).sum())
    false_positive = int((predicted_positive & y_true.eq(0)).sum())
    false_negative = int((~predicted_positive & y_true.eq(1)).sum())
    true_negative = int((~predicted_positive & y_true.eq(0)).sum())
    sensitivity = true_positive / (true_positive + false_negative)
    false_positive_rate = false_positive / (false_positive + true_negative)
    specificity = 1 - false_positive_rate
    return {
        "cutoff": cutoff,
        "TP": true_positive,
        "FP": false_positive,
        "FN": false_negative,
        "TN": true_negative,
        "sensitivity_TPR": sensitivity,
        "false_positive_rate_FPR": false_positive_rate,
        "specificity": specificity,
        "youden_J": sensitivity + specificity - 1,
        "precision": true_positive / (true_positive + false_positive)
        if true_positive + false_positive
        else np.nan,
    }


optimal_metrics = cutoff_metrics(float(optimal_cutoff))
proposed_metrics = cutoff_metrics(446)
previous_metrics = cutoff_metrics(491)
print("Youden cutoff fit to historical worst-debt outcome (group >= 3)")
print(f"Observed customers: {len(threshold_data):,}; positive cases: {int(y_true.sum()):,}")
print(pd.DataFrame([optimal_metrics, proposed_metrics, previous_metrics], index=["Youden-optimal cutoff", "Proposed model cutoff 446", "Previous cutoff 491 (comparison)"]).round(3).to_string())
print(f"\nMaximum Youden J: {youden_values[optimal_index]:.3f}")
print("Caution: these are in-sample metrics used to select the cutoff; they are optimistic and require out-of-time validation.")
print("This optimizes classification of observed historical debt group >=3, not verified future default.")
